# COMP — Comparaison cohérence SIREN ↔ SIRET

Produit deux fichiers Excel montrant la cohérence entre les deux pipelines,
organisée par couple (personne morale, établissement).

**Source des données** : les 3 fichiers de phase de chaque pipeline, sans passer
par une consolidation intermédiaire. La fusion se fait à la volée :
- Phase 1 : toutes les lignes
- Phase 2 (rang 1) : celles non validées en P1
- Phase 3 (rang 1) : celles validées ni en P1 ni en P2

**Logique de cohérence** : pour chaque EGE validé, on joint sa PM parente via
`NB_PmSmsseId` avec le résultat de la sirenisation, puis on compare
`SIRET_EGE[:9]` avec le `SIREN_PM` retenu.

**Statuts produits**
- **COHERENT** : EGE validé + PM validée + `SIRET_EGE[:9] == SIREN_PM`
- **INCOHERENT** : EGE validé + PM validée + identifiants divergents
- **PARTIEL_EG** : EGE validé mais PM parente non validée
- **PARTIEL_EJ** : PM validée mais aucun EGE validé associé
- **ORPHELIN** : EGE validé sans `NB_PmSmsseId` renseigné

**Fichiers produits**
- `coherence_globale.xlsx` — Synthese / Coherent / Incoherent / Partiel
- `coherence_par_phase.xlsx` — découpage des cohérents par couple de phases

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.comparaison  import (
    charger_siretisation, charger_sirenisation,
    construire_vue_coherence, synthese_globale,
)
from src.excel_export import export_comparaison_excel
from src.display      import afficher_tableau
from config.settings  import (
    ST_PHASE1, ST_PHASE2, ST_PHASE3,
    SN_PHASE1, SN_PHASE2, SN_PHASE3,
    COMP_GLOBALE, RESULTS_COMP_DIR,
)

RESULTS_COMP_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement et fusion des 3 fichiers de phase

In [2]:
df_st = charger_siretisation(ST_PHASE1, ST_PHASE2, ST_PHASE3)
df_sn = charger_sirenisation(SN_PHASE1, SN_PHASE2, SN_PHASE3)

print(f'Siretisation (fusion P1 + P2 + P3) : {len(df_st):,} lignes')
print(df_st["statut"].value_counts().to_string())
print(f'\nSirenisation (fusion P1 + P2 + P3) : {len(df_sn):,} lignes')
print(df_sn["statut"].value_counts().to_string())

Siretisation (fusion P1 + P2 + P3) : 160,686 lignes
statut
VALIDE            64779
VALIDE_FORT       29676
DOUTEUX           25429
SANS_SIRET        13104
SIRET_INCONNU     12242
REJETE            10581
NON_FIABLE_APE     4650
SANS_CANDIDAT       225

Sirenisation (fusion P1 + P2 + P3) : 77,077 lignes
statut
VALIDE           27900
VALIDE_FORT      25929
DOUTEUX          12563
SIREN_INCONNU     5604
REJETE            2984
SANS_SIREN        2090
SANS_CANDIDAT        7


In [3]:
import re

def _norm(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return re.sub(r"\s", "", str(v).strip())

VALIDES = {"VALIDE_FORT", "VALIDE"}
df_ege_v = df_st[df_st['statut'].isin(VALIDES)]
df_pm_v  = df_sn[df_sn['statut'].isin(VALIDES)]

ids_ege_pointent_vers = set(df_ege_v['NB_PmSmsseId'].dropna().apply(_norm))
ids_ege_pointent_vers.discard("")
ids_pm_dispos = set(df_pm_v['NB_PmSmsseId'].dropna().apply(_norm))
ids_pm_dispos.discard("")

print(f"NB_PmSmsseId distincts cote EGE (df_st) : {len(ids_ege_pointent_vers):,}")
print(f"NB_PmSmsseId distincts cote PM  (df_sn) : {len(ids_pm_dispos):,}")
print(f"Intersection                            : {len(ids_ege_pointent_vers & ids_pm_dispos):,}")
print()
print("Echantillon cote EGE :", list(ids_ege_pointent_vers)[:5])
print("Echantillon cote PM  :", list(ids_pm_dispos)[:5])

NB_PmSmsseId distincts cote EGE (df_st) : 46,859
NB_PmSmsseId distincts cote PM  (df_sn) : 53,829
Intersection                            : 45,948

Echantillon cote EGE : ['64052', '28497', '30196', '40028', '77038']
Echantillon cote PM  : ['64052', '28497', '10489', '30196', '40028']


## 2. Synthèse globale Siretisation × Sirenisation

In [4]:
df_synth = synthese_globale(df_st, df_sn)
afficher_tableau(df_synth, 'Synthese globale Siretisation x Sirenisation', max_lignes=25)

Indicateur,Siretisation,Sirenisation
VALIDE_FORT (Phase 1),24600,23085
VALIDE_FORT (Phase 2),5075,2841
VALIDE_FORT (Phase 3),1,3
VALIDE (Phase 1),35948,15700
VALIDE (Phase 2),27090,11091
VALIDE (Phase 3),1741,1109
—,,
DOUTEUX,25429,12563
REJETE,10581,2984
SANS_CANDIDAT,225,7


## 3. Construction de la vue des couples (EGE, PM)

In [5]:
df_vue = construire_vue_coherence(df_st, df_sn)
print(f'Lignes totales : {len(df_vue):,}')
print()
print(df_vue['statut_coherence'].value_counts().to_string())

Lignes totales : 102,336

statut_coherence
COHERENT      79474
INCOHERENT    13669
PARTIEL_EJ     7881
PARTIEL_EG     1312


## 4. Aperçu des couples cohérents

In [6]:
df_coh = df_vue[df_vue['statut_coherence'] == 'COHERENT']
afficher_tableau(
    df_coh[['finess_pm', 'nom_pm', 'siren_pm_retenu',
            'finess_ege', 'nom_ege', 'siret_ege_retenu', 'siren_du_siret_ege']],
    f'Couples coherents ({len(df_coh):,} lignes)', max_lignes=10,
)

finess_pm,nom_pm,siren_pm_retenu,finess_ege,nom_ege,siret_ege_retenu,siren_du_siret_ege
010780054,CH FLEYRIAT,260100045,010000024,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,26010004500012,260100045
010780062,CH BUGEY SUD,260100037,010000032,CENTRE HOSPITALIER BUGEY SUD,26010003700068,260100037
010780096,CH DE TREVOUX,260100284,010000065,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,26010028400017,260100284
010780112,CH DU PAYS DE GEX,260100102,010000081,CENTRE HOSPITALIER DU PAYS DE GEX,26010010200011,260100102
010002277,NOUVELLE PHARMACIE DU CHAMP DE MARS,893939983,010002285,NOUVELLE PHARMACIE DU CHAMP DE MARS,89393998300019,893939983
010002327,PHARMACIE ORSET,409526704,010002335,PHARMACIE ORSET,40952670400039,409526704
010002343,PHARMACIE DE LA GARE,340555879,010002350,PHARMACIE DE LA GARE,34055587900014,340555879
010002392,PHARMACIE D'AMBRONAY,803684679,010002400,PHARMACIE D'AMBRONAY,80368467900027,803684679
690039904,ENSEIS - SIEGE SOCIAL,392694014,010002459,ENSEIS - ÉTABLISSEMENT DE L'AIN,39269401400102,392694014
010002442,SELARL TONY MENU,849124318,010002467,PHARMACIE D'ARS ET DOMBES,84912431800015,849124318


## 5. Aperçu des incohérences (à arbitrer)

In [7]:
df_incoh = df_vue[df_vue['statut_coherence'] == 'INCOHERENT']
if len(df_incoh) > 0:
    afficher_tableau(
        df_incoh[['finess_pm', 'nom_pm', 'siren_pm_retenu',
                  'finess_ege', 'nom_ege', 'siret_ege_retenu', 'siren_du_siret_ege']],
        f'Couples incoherents ({len(df_incoh):,} lignes)', max_lignes=10,
    )
else:
    print('Aucun cas incoherent.')

finess_pm,nom_pm,siren_pm_retenu,finess_ege,nom_ege,siret_ege_retenu,siren_du_siret_ege
010000222,CLINIQUE DU SOUFFLE LE PONTET,503330615,010011641,CLINIQUE DU SOUFFLE LE PONTET,54612012200033,546120122
010011765,YOUCARE BOURG,490654688,010011773,SAAD YOUCARE BOURG,83501086900039,835010869
010784841,CCAS MARBOZ,799381405,010784643,RESIDENCE FOYER SOLEIL MARBOZ,26011004400021,260110044
750054157,OPPELIA,382656981,020006292,CSAPA HORIZON 02 - ASSOCIATION OPPELIA,32602117700406,326021177
020006318,FEDERATION ADMR DE L'AISNE,780198644,020018768,SAD ADMR FAMILLE ENFANCE PARENTALITE,31014732700046,310147327
800022261,AUXILAIR PICARDIE,481076636,020019212,AUXILAIR PICARDIE SAINT QUENTIN,88423872600027,884238726
940020175,INFA,431859818,030007579,INSTITUT NATIONAL DE FORMATION ET D'APPLICATION DE L'ALLIER,78574029100915,785740291
030004949,FEDERATION ADMR DE L'ALLIER,478465198,030009609,SAAD ASSOCIATION ADMR LE MONTET,78905752800018,789057528
030004329,ASSOC. MAISON DE RETRAITE L'ERMITAGE,438824427,030782643,MAISON DE RETRAITE L'ERMITAGE,42511166300026,425111663
040786360,FEDERATION ADMR ALPES HTE PROVENCE,308973965,040005431,SAAD ADMR DE CASTELLANE,38984914200044,389849142


## 6. Export Excel (4 feuilles : Synthese / Coherent / Incoherent / Partiel)

In [8]:
compteurs = export_comparaison_excel(df_vue, COMP_GLOBALE)

print('Synthese de la coherence :')
print(f'  PM avec tous EGE coherents     : {compteurs["EJ_tous_coherents"]:,}')
print(f'  PM avec au moins un incoherent : {compteurs["EJ_avec_incoherent"]:,}')
print(f'  PM validees sans EGE valide    : {compteurs["PARTIEL_EJ"]:,}')
print(f'  EGE valides sans PM validee    : {compteurs["PARTIEL_EG"]:,}')
print(f'  EGE orphelins (sans PM)        : {compteurs["ORPHELIN"]:,}')
print()
print(f'  Total couples coherents   : {compteurs["COHERENT_couples"]:,}')
print(f'  Total couples incoherents : {compteurs["INCOHERENT_couples"]:,}')
print(f'\nFichier : {COMP_GLOBALE}')

Synthese de la coherence :
  PM avec tous EGE coherents     : 39,946
  PM avec au moins un incoherent : 6,002
  PM validees sans EGE valide    : 7,881
  EGE valides sans PM validee    : 1,312
  EGE orphelins (sans PM)        : 0

  Total couples coherents   : 79,474
  Total couples incoherents : 13,669

Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/comparaison/coherence_globale.xlsx


---
## Séparation de comparaison par phase

Découpe les couples cohérents selon la phase qui a validé chaque côté :
P1/P1, P2/P2, P3/P3, et phases mixtes quand les deux côtés n'ont pas été validés
à la même phase.

Le découpage travaille sur `df_vue` en mémoire plutôt que de relire le fichier
Excel : le résultat est le même et la chaîne ne dépend pas des noms de feuilles.

#### Découpage

In [9]:
from src.comparaison  import decouper_par_phase, synthese_par_phase
from src.excel_export import export_coherence_par_phase
from config.settings  import COMP_PAR_PHASE

blocs = decouper_par_phase(df_vue)

for nom, bloc in blocs.items():
    print(f'{nom:<16} {len(bloc):>9,}')

Coherent_P1         55,225
Coherent_P2          6,051
Coherent_P3            130
Coherent_mixte      18,068
Incoherent          13,669
Partiel_PM           7,881
Partiel_EGE          1,312


#### Synthèse

In [10]:
couples, structures = synthese_par_phase(blocs)

afficher_tableau(couples, 'Couples par categorie', max_lignes=15)
afficher_tableau(structures, 'Structures distinctes par categorie', max_lignes=15)

Categorie,Couples
Coherent - phase 1,55225
Coherent - phase 2,6051
Coherent - phase 3,130
Coherent - phases mixtes,18068
TOTAL COHERENTS,79474
Incoherent,13669
Partiel PM (PM seule validee),7881
Partiel EGE (EGE seul valide),1312


Categorie,Distinctes
PM coherentes en phase 1,31990
PM coherentes en phase 2,5566
PM coherentes en phase 3,118
PM coherentes phases mixtes,7427
TOTAL PM coherentes,42234
PM avec au moins un EGE incoherent,6002
PM sans EGE valide,7881
EGE sans PM validee,1312


#### Export

In [11]:
export_coherence_par_phase(blocs, couples, structures, COMP_PAR_PHASE)
print(f'Fichier : {COMP_PAR_PHASE}')

Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/comparaison/coherence_par_phase.xlsx
